Author: Bryan Sandor

Course: Stat 554

Title: Final Project

# Preamble

For this project, we'll use a `Jupyter` notebook fitting a machine learning model using `pyspark`'s `MLib` module. We'll use code to read in a stream of data and use that model to perform predictions on the stream, outputing them to the console. The data is modified from the UCI machine learning repository and studies the relationship between power consumption in Tetouan City and various factors like time of day, temperature, and humidity.

We begin by importing all the necessary libraries.

In [29]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml.feature import SQLTransformer, Binarizer, OneHotEncoder, VectorAssembler, PCA
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator
import pyspark.sql.functions as F

# Part 1: Fitting the Model

## Reading in the Data

We will read the data in using a `pandas` data frame, initially.

In [2]:
powerData = pd.read_csv("power_ml_data.csv")

Then we convert it to a `spark` data frame, first initializing our `spark` session.

In [3]:
spark = SparkSession.builder \
        .appName("proj2") \
        .config("spark.sql.ansi.enabled", "false") \
        .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/29 21:06:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/29 21:06:01 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [4]:
powerDF = spark.createDataFrame(powerData)
powerDF.show(n = 10, truncate = False)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|6.559      |73.8    |0.083     |0.051                |0.119        |34055.6962  |16128.87538 |20240.96386 |1    |0   |
|6.414      |74.5    |0.083     |0.07                 |0.085        |29814.68354 |19375.07599 |20131.08434 |1    |0   |
|6.313      |74.5    |0.08      |0.062                |0.1          |29128.10127 |19006.68693 |19668.43373 |1    |0   |
|6.121      |75.0    |0.083     |0.091                |0.096        |28228.86076 |18361.09422 |18899.27711 |1    |0   |
|5.921      |75.7    |0.081     |0.048                |0.085        |27335.6962  |17872.34043 |18442.40964 |1    |0   |
|5.853      |76.9    |0.081     |0.059  

For this assignment, we will use `Power_Zone_3` as the response and the other variables as predictors.

## Pipeline: Manipulations and Formatting

Now we need to make some adjustments to the data.

### Change `Hour` Class

In [5]:
powerDF.schema

StructType([StructField('Temperature', DoubleType(), True), StructField('Humidity', DoubleType(), True), StructField('Wind_Speed', DoubleType(), True), StructField('General_Diffuse_Flows', DoubleType(), True), StructField('Diffuse_Flows', DoubleType(), True), StructField('Power_Zone_1', DoubleType(), True), StructField('Power_Zone_2', DoubleType(), True), StructField('Power_Zone_3', DoubleType(), True), StructField('Month', LongType(), True), StructField('Hour', LongType(), True)])

First, we need to re-cast the `Hour` column as `DoubleType` instead of `LongType()`.

In [6]:
recast = SQLTransformer(statement="SELECT *, CAST(Hour AS DOUBLE) AS db_hour FROM __THIS__")

recast.transform(powerDF).schema

StructType([StructField('Temperature', DoubleType(), True), StructField('Humidity', DoubleType(), True), StructField('Wind_Speed', DoubleType(), True), StructField('General_Diffuse_Flows', DoubleType(), True), StructField('Diffuse_Flows', DoubleType(), True), StructField('Power_Zone_1', DoubleType(), True), StructField('Power_Zone_2', DoubleType(), True), StructField('Power_Zone_3', DoubleType(), True), StructField('Month', LongType(), True), StructField('Hour', LongType(), True), StructField('db_hour', DoubleType(), True)])

### Binarize `Hour` Column

Now that the `Hour` column is appropriately cast, we binarize it with the margin at $6.5$, i.e., night vs day.

In [7]:
binaryTrans = Binarizer(threshold = 6.5, inputCol = "db_hour", outputCol = "bin_hour")

binaryTrans.transform(
    recast.transform(powerDF)
).show()

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-------+--------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|db_hour|bin_hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-------+--------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|    0.0|     0.0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|    0.0|     0.0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|    0.0|     0.0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|    0.0|     0.0|
|      5.921|    75.7|     0.081|        

### One-Hot Encode the `Month` Column

Next, we will one-hot encode the `Month` column

In [8]:
encodeTrans = OneHotEncoder(inputCol = "Month", outputCol = "enc_month")

encodeTrans.fit(powerDF).transform(
    binaryTrans.transform(
        recast.transform(powerDF)
    )
).show()

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-------+--------+--------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|db_hour|bin_hour|     enc_month|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-------+--------+--------------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|    0.0|     0.0|(12,[1],[1.0])|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|    0.0|     0.0|(12,[1],[1.0])|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|    0.0|     0.0|(12,[1],[1.0])|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361

### PCA fit

To perform a PCA fit, we need to map several columns into a single column, which we will accomplish using the `VectorAssembler()`, combining `Temperature`, `Humidity`, `Wind_Speed`, `General_Diffuse_Flows`, and `Diffuse_Flows`.

In [9]:
assemblerPCA = VectorAssembler(inputCols = ["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows", "Diffuse_Flows"], outputCol = "PC")

In [10]:
assemblerPCA.transform(
    encodeTrans.fit(powerDF).transform(
    binaryTrans.transform(
        recast.transform(powerDF)
    )
    )
).show()

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-------+--------+--------------+--------------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|db_hour|bin_hour|     enc_month|                  PC|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-------+--------+--------------+--------------------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|    0.0|     0.0|(12,[1],[1.0])|[6.559,73.8,0.083...|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|    0.0|     0.0|(12,[1],[1.0])|[6.414,74.5,0.083...|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|    0.0|     0.0

Now we perform the PCA fit.

In [11]:
pca = PCA(k = 2, inputCol = "PC")
pca.setOutputCol("pca_features")
pcaTrans = pca.fit(
    assemblerPCA.transform(
    encodeTrans.fit(powerDF).transform(
    binaryTrans.transform(
        recast.transform(powerDF)
    )
    )
    )
)

In [12]:
pcaTrans.transform(
    assemblerPCA.transform(
    encodeTrans.fit(powerDF).transform(
    binaryTrans.transform(
        recast.transform(powerDF)
    )
    )
    )
).show()

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-------+--------+--------------+--------------------+--------------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|db_hour|bin_hour|     enc_month|                  PC|        pca_features|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-------+--------+--------------+--------------------+--------------------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|    0.0|     0.0|(12,[1],[1.0])|[6.559,73.8,0.083...|[1.79440486365695...|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|    0.0|     0.0|(12,[1],[1.0])|[6.414,74.5,0.083...|[1.80604083009823...|
|      6.313|    74.5|      0.

### Response and Predictors

To prepare the data for analysis, we begin by setting `Power_Zone_3` as our response by renaming it `label`.

In [13]:
labelTrans = SQLTransformer(
    statement = """
                SELECT pca_features, bin_hour, Power_Zone_1, Power_Zone_2, enc_month, Power_Zone_3 as label FROM __THIS__
                """
)

labelTrans.transform(
    pcaTrans.transform(
    assemblerPCA.transform(
    encodeTrans.fit(powerDF).transform(
    binaryTrans.transform(
        recast.transform(powerDF)
    )
    )
    )
    )
).show()

+--------------------+--------+------------+------------+--------------+-----------+
|        pca_features|bin_hour|Power_Zone_1|Power_Zone_2|     enc_month|      label|
+--------------------+--------+------------+------------+--------------+-----------+
|[1.79440486365695...|     0.0|  34055.6962| 16128.87538|(12,[1],[1.0])|20240.96386|
|[1.80604083009823...|     0.0| 29814.68354| 19375.07599|(12,[1],[1.0])|20131.08434|
|[1.81022976305639...|     0.0| 29128.10127| 19006.68693|(12,[1],[1.0])|19668.43373|
|[1.79866765174088...|     0.0| 28228.86076| 18361.09422|(12,[1],[1.0])|18899.27711|
|[1.86328720163797...|     0.0|  27335.6962| 17872.34043|(12,[1],[1.0])|18442.40964|
|[1.87820674500461...|     0.0| 26624.81013| 17416.41337|(12,[1],[1.0])|18130.12048|
|[1.91529298717955...|     0.0| 25998.98734| 16993.31307|(12,[1],[1.0])|17945.06024|
|[1.92400540807029...|     0.0| 25446.07595| 16661.39818|(12,[1],[1.0])|17459.27711|
|[1.89501930353027...|     0.0| 24777.72152| 16227.35562|(12,[1],

Finally, we combine the predictors into a single column named `features`.

In [14]:
assemblerFeatures = VectorAssembler(inputCols = ["pca_features", "bin_hour", "Power_Zone_1", "Power_Zone_2", "enc_month"], outputCol = "features")

assemblerFeatures.transform(
    labelTrans.transform(
    pcaTrans.transform(
    assemblerPCA.transform(
    encodeTrans.fit(powerDF).transform(
    binaryTrans.transform(
        recast.transform(powerDF)
    )
    )
    )
    )
    )
).show(truncate = False)

+----------------------------------------+--------+------------+------------+--------------+-----------+-------------------------------------------------------------------------------------+
|pca_features                            |bin_hour|Power_Zone_1|Power_Zone_2|enc_month     |label      |features                                                                             |
+----------------------------------------+--------+------------+------------+--------------+-----------+-------------------------------------------------------------------------------------+
|[1.7944048636569547,-0.7412746447404446]|0.0     |34055.6962  |16128.87538 |(12,[1],[1.0])|20240.96386|(17,[0,1,3,4,6],[1.7944048636569547,-0.7412746447404446,34055.6962,16128.87538,1.0]) |
|[1.8060408300982318,-0.7108534239558469]|0.0     |29814.68354 |19375.07599 |(12,[1],[1.0])|20131.08434|(17,[0,1,3,4,6],[1.8060408300982318,-0.7108534239558469,29814.68354,19375.07599,1.0])|
|[1.8102297630563908,-0.7283113191158928]|0.0

## Elastic Net Model

Given specific parameters, we will use cross validation and linear regression to fit an elastic net model with our previous transformations in a pipeline.

In [15]:
lr = LinearRegression(standardization = True)

paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()

pipeline_lm = Pipeline(stages = [recast, binaryTrans, encodeTrans, assemblerPCA, pcaTrans, labelTrans, assemblerFeatures, lr])

At the suggestion of a classmate, we will make use of the parallel processing for a $5$-fold cross validation (using the default root mean squared error (RMSE)).

In [16]:
crossval_lm = CrossValidator(estimator = pipeline_lm,
                             estimatorParamMaps = paramGrid,
                             evaluator = RegressionEvaluator(),
                             parallelism = 128,
                             seed = 1982,
                             numFolds = 5)

Finally we fit the model.

In [17]:
cvLinModel = crossval_lm.fit(powerDF)

26/04/29 21:09:39 ERROR LBFGS: Failure! Resetting history: breeze.optimize.FirstOrderException: Line search zoom failed
26/04/29 21:09:39 ERROR LBFGS: Failure again! Giving up and returning. Maybe the objective is just poorly behaved?
26/04/29 21:09:43 ERROR LBFGS: Failure! Resetting history: breeze.optimize.FirstOrderException: Line search zoom failed
26/04/29 21:09:44 ERROR LBFGS: Failure! Resetting history: breeze.optimize.FirstOrderException: Line search zoom failed


We print out all $11 \times 11 = 121$ combinations.

In [18]:
my_list = []
for i in range(len(paramGrid)):
    my_list.append([cvLinModel.avgMetrics[i], paramGrid[i].values()])
my_list

[[np.float64(2147.547722631704), dict_values([0.0, 0.0])],
 [np.float64(2147.547723473428), dict_values([0.0, 0.05])],
 [np.float64(2147.5477226317043), dict_values([0.0, 0.1])],
 [np.float64(2147.5477225608906), dict_values([0.0, 0.25])],
 [np.float64(2147.547722631704), dict_values([0.0, 0.5])],
 [np.float64(2147.547722631705), dict_values([0.0, 0.75])],
 [np.float64(2147.5477226317043), dict_values([0.0, 0.9])],
 [np.float64(2147.5477226317053), dict_values([0.0, 0.95])],
 [np.float64(2147.547722631704), dict_values([0.0, 0.98])],
 [np.float64(2147.5477232063176), dict_values([0.0, 0.99])],
 [np.float64(2147.5477226317043), dict_values([0.0, 1.0])],
 [np.float64(2147.547741406358), dict_values([0.05, 0.0])],
 [np.float64(2147.5482890271114), dict_values([0.05, 0.05])],
 [np.float64(2147.5473932101604), dict_values([0.05, 0.1])],
 [np.float64(2147.5477102047234), dict_values([0.05, 0.25])],
 [np.float64(2147.548124452775), dict_values([0.05, 0.5])],
 [np.float64(2147.546907066976), d

Then we seek the parameters for the best model.

In [43]:
bestLModel = cvLinModel.bestModel.stages[-1]
bestLModel.extractParamMap()

{Param(parent='LinearRegression_e943b5cb6fc6', name='aggregationDepth', doc='suggested depth for treeAggregate (>= 2).'): 2,
 Param(parent='LinearRegression_e943b5cb6fc6', name='elasticNetParam', doc='the ElasticNet mixing parameter, in range [0, 1]. For alpha = 0, the penalty is an L2 penalty. For alpha = 1, it is an L1 penalty.'): 0.25,
 Param(parent='LinearRegression_e943b5cb6fc6', name='epsilon', doc='The shape parameter to control the amount of robustness. Must be > 1.0. Only valid when loss is huber'): 1.35,
 Param(parent='LinearRegression_e943b5cb6fc6', name='featuresCol', doc='features column name.'): 'features',
 Param(parent='LinearRegression_e943b5cb6fc6', name='fitIntercept', doc='whether to fit an intercept term.'): True,
 Param(parent='LinearRegression_e943b5cb6fc6', name='labelCol', doc='label column name.'): 'label',
 Param(parent='LinearRegression_e943b5cb6fc6', name='loss', doc='The loss function to be optimized. Supported options: squaredError, huber.'): 'squaredErro

The key lines in this printout concern the `elasticNetParam`, which for the best model is $\alpha = 0.25$, and the `regParam`, which is $\lambda = 0.1$. We then find the error associated with these parameters, ie., the minimum error through cross validation.

In [44]:
CVError = min(cvLinModel.avgMetrics)
CVError

np.float64(2147.5468555075277)

This may be verified by parsing the above table, most easily done via `Ctrl+F`. Using the entire dataset as the training set we perform predictions using the model.

In [27]:
cvLinModel.transform(powerDF).select("label", "prediction").show()

+-----------+------------------+
|      label|        prediction|
+-----------+------------------+
|20240.96386|20880.339701309276|
|20131.08434|18659.921478082164|
|19668.43373|18204.427090266247|
|18899.27711|17590.347680928135|
|18442.40964|16996.978968118237|
|18130.12048|16517.366359426887|
|17945.06024|16092.934692961886|
|17459.27711| 15722.37908101309|
|17025.54217|15270.731694846889|
|16794.21687|14938.028946616034|
|16638.07229|14652.143575262055|
|16395.18072| 14414.66076882506|
|16117.59036|14082.559411468794|
| 15822.6506|13624.562955559124|
|15672.28916|13450.063852264599|
|15597.10843|13302.029790063645|
|15510.36145|13154.609299214311|
|15336.86747|12994.976789618458|
|15140.24096|  12758.6691989663|
|15059.27711|12674.752406059693|
+-----------+------------------+
only showing top 20 rows


And then we find the RMSE for the model.

In [28]:
train_error_lm = RegressionEvaluator().evaluate(cvLinModel.transform(powerDF))
print(train_error_lm)

2147.0973812905354


Lastly, we compare the actual values, the predicted values, and examine their residuals (difference between them).

In [31]:
resultsPowerDF = cvLinModel.transform(powerDF).withColumn("residuals", F.col("label") - F.col("prediction"))

In [34]:
resultsPowerDF.select("label", "prediction", "residuals").show()

+-----------+------------------+------------------+
|      label|        prediction|         residuals|
+-----------+------------------+------------------+
|20240.96386|20880.339701309276|-639.3758413092764|
|20131.08434|18659.921478082164|1471.1628619178373|
|19668.43373|18204.427090266247|1464.0066397337541|
|18899.27711|17590.347680928135|1308.9294290718644|
|18442.40964|16996.978968118237|1445.4306718817643|
|18130.12048|16517.366359426887| 1612.754120573114|
|17945.06024|16092.934692961886| 1852.125547038113|
|17459.27711| 15722.37908101309|1736.8980289869087|
|17025.54217|15270.731694846889|1754.8104751531118|
|16794.21687|14938.028946616034|1856.1879233839663|
|16638.07229|14652.143575262055|1985.9287147379455|
|16395.18072| 14414.66076882506|1980.5199511749397|
|16117.59036|14082.559411468794|2035.0309485312064|
| 15822.6506|13624.562955559124| 2198.087644440877|
|15672.28916|13450.063852264599|2222.2253077354017|
|15597.10843|13302.029790063645| 2295.078639936355|
|15510.36145

# Part Two: Streaming Data